# Video Game Commercial Success Prediction & Market Analysis

The goal of this project is to deliver a clear commercial narrative that game studios and publishers can use to minimize financial risk before greenlighting a new game's production.

When looking into creating new games studios and publishers investigate what has been historically successful. By analyzing characteristics of games that have already been released, we can help guide new game ideas to the right publishers to help propagate the game to a better sales pattern, allowing both the publisher and game developer to maximize their investment.

#### 1. Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# SQL alchemy
from sqlalchemy import create_engine, text

# scikitlearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [ ]:
# create in-memory SQLite database
engine = create_engine('sqlite:///:memory:')

In [ ]:
# load raw data
df_raw = pd.read_csv('datasets/Video_Games_Sales.csv')

In [ ]:
# ingest raw data into database
df_raw.to_sql('raw_game_sales', con=engine, index=False, if_exists='replace')

In [ ]:
# run wuery against database to confirm ingestion
query = """
SELECT count(*)
FROM raw_game_sales
"""

In [ ]:
with engine.connect() as conn:
    results = conn.execute(text(query))
    print(f'Successfully ingested {results.scalar()} records into in-memory table "raw_game_sales".')

In [ ]:
# pull data from database to pandas DataFrame
query = """
SELECT platform as platform,
    year_of_release as release_year,
    genre as genre,
    publisher as publisher,
    global_sales as global_sales
FROM raw_game_sales
WHERE global_sales IS NOT NULL
AND year_of_release IS NOT NULL
AND publisher IS NOT NULL;
"""

In [ ]:
with engine.connect() as conn:
    results = conn.execute(text(query))
    df_pulled = pd.DataFrame(results)

In [ ]:
df_pulled.info()

#### 2. Data Cleaning & Target Creation

In [ ]:
# drop missing values
df_clean = df_pulled.dropna(subset='genre').copy()

In [ ]:
# parse release_year to 'int' from 'float64'
df_clean['release_year'] = df_clean['release_year'].astype(int)

##### Create Dynamic Target

In [ ]:
# top 20% global sales within each release year
df_clean['year_80th_p'] = df_clean.groupby('release_year')['global_sales'].transform(lambda x: x.quantile(0.80))
df_clean['is_hit'] = (df_clean['global_sales'] >= df_clean['year_80th_p']).astype(int)

In [ ]:
# reduction of publisher cardinality
top_publishers = df_clean['publisher'].value_counts().nlargest(15).index
df_clean['publisher_grouped'] = df_clean['publisher'].apply(lambda x: x if x in top_publishers else 'Other')

In [ ]:
print(f'Cleaned dataset shape: {df_clean.shape}')
print(f'Target Distribution (is_hit): \n', df_clean['is_hit'].value_counts(normalize=True))

#### 3. Feature Selection & Train/Test Split

In [ ]:
# Strictly pre-release features to prevent target leakage
features = ['platform', 'genre', 'publisher_grouped', 'release_year']

X = df_clean[features]
y = df_clean['is_hit']

In [ ]:
# split to preserve class ratios
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42, stratify = y
)

#### 4. Preprocessing & Model Pipline

In [ ]:
categorical_features = ['platform', 'genre', 'publisher_grouped']
numeric_features = ['release_year']

In [ ]:
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False), categorical_features)
    ]
)

In [ ]:
pipeline = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators = 100, random_state = 42))
])

In [ ]:
# fit model on training set
pipeline.fit(X_train, y_train)

#### 5. Evaluation

In [ ]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

In [ ]:
print('--- Model Evaluation ---')
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}')
print(f'\nClassification report:\n', classification_report(y_test, y_pred))